In [1]:
!pip install faker

Defaulting to user installation because normal site-packages is not writeable


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Auvik

In [8]:
import json
import random
import base64
from datetime import datetime, timedelta, timezone

def encode_auvik_id(company_id, object_id):
    raw = f"{company_id},{object_id}".encode()
    return base64.b64encode(raw).decode().rstrip("=")

def format_auvik_ts(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

def make_object_id():
    return str(fake.random_number(digits=19, fix_len=True))

customers = [
    {
        "name": "Ambient Enterprises DMG San Luis Obispo",
        "slug": "ambiententerprisesdmgslo",
        "site": "DMG-SLO",
        "companyId": "1083819129238950653",
        "templates": ["ap_offline"],
    },
    {
        "name": "DYOPATH Oakbrook Terrace",
        "slug": "dyopathoakbrookterrace",
        "site": "dyoOBT",
        "companyId": "1167420203531866877",
        "templates": ["interface_mismatch"],
    },
    {
        "name": "HVP SLHV Data Center",
        "slug": "hvpslhvdatacenter",
        "site": "slhvDC",
        "companyId": "1139449271987278589",
        "templates": ["vpn_gateway_lost"],
    },
    {
        "name": "Hanover Company Azure",
        "slug": "hanovercompanyazure",
        "site": "HAN-AZURE",
        "companyId": "748793047432794877",
        "templates": ["vpn_gateway_lost"],
    },
    {
        "name": "DYOPATH Azure Central",
        "slug": "dyopathazurecentral",
        "site": "AZC",
        "companyId": "860416283769541373",
        "templates": ["vpn_gateway_lost"],
    },
    {
        "name": "DYOPATH Oakbrook Terrace Lab/Depot",
        "slug": "dyopathoakbrookterracelab",
        "site": "dpOBT",
        "companyId": "1175016027093177085",
        "templates": ["internet_lost", "network_element_offline"],
    },
    {
        "name": "HVP SLHV South County",
        "slug": "hvpslhvsouthcounty",
        "site": "slhvSC",
        "companyId": "1294184548170142461",
        "templates": ["network_element_offline"],
    },
]

entity_registry = {}

def get_entity_numeric_id(company_id, entity_name):
    key = (company_id, entity_name)
    if key not in entity_registry:
        entity_registry[key] = make_object_id()
    return entity_registry[key]

def build_ap_offline(customer):
    entity_name = f"{customer['site']}-AP{random.randint(1, 4):02d}"
    model = random.choice(["Ubiquiti U6+", "Meraki MR46", "Meraki MR44"])
    return {
        "alertName": "Access Point Offline",
        "entityType": "device",
        "entityName": entity_name,
        "alertSeverityString": "Warning",
        "alertSeverity": 3,
        "triggerDescription": f"This network element has gone offline   {model}",
        "clearDescription": "Device Online Status is now equal to Online",
    }

def build_interface_mismatch(customer):
    host = f"{customer['site']}extsw{random.randint(1, 2):02d}.dyopath.com"
    port = f"GigabitEthernet1/0/{random.randint(1, 48)}"
    connection = random.choice([
        "Connection to Lab-Depot ASAx Outside Gi1/1",
        "Uplink to Core Switch",
        "WAN handoff",
        "Connection to firewall outside interface",
    ])
    peer_interface = random.choice(["wan1", "wan2", "Gi1/1", "port10"])
    peer_host = random.choice(["dyo-sch-fw01a", "dyo-core-fw01", "dpOBTfw01.dyopathcorp.com"])

    return {
        "alertName": "Interface Status Mismatch",
        "entityType": "interface",
        "entityName": f"{port} on {host}",
        "alertSeverityString": "Warning",
        "alertSeverity": 3,
        "triggerDescription": (
            f"Interface {port} ({connection}) on {host} is administratively Up and has gone Down. "
            f"It's connected to {peer_interface} on {peer_host}."
        ),
        "clearDescription": (
            f"The operational status for interface {port} ({connection}) on {host} is now Up and "
            f"its administrative status is now Up. It's connected to {peer_interface} on {peer_host}."
        ),
    }

def build_vpn_gateway_lost(customer):
    entity_name = random.choice([
        "asa5515x-01.kslinc.net",
        "HAN-AZURE-ASAV",
        "asav-02.dyopathcorp.com",
        "asa-edge-01.corp.local",
    ])
    gateway_ip = random.choice(["12.207.114.35", "12.207.114.41", fake.ipv4()])

    return {
        "alertName": "VPN remote gateway is Lost",
        "entityType": "service",
        "entityName": entity_name,
        "alertSeverityString": "Critical",
        "alertSeverity": 2,
        "triggerDescription": (
            f"The VPN remote gateway {gateway_ip} has been lost\r\n\r\n"
            "Begin Crisis Management\r\n"
            "Warm Hand-off to IS - Networking team"
        ),
        "clearDescription": f"The VPN remote gateway {gateway_ip} is now reachable",
    }

def build_internet_lost(customer):
    fw_host = f"{customer['site']}fw{random.randint(1, 2):02d}.dyopathcorp.com"
    gateway_ip = random.choice(["12.207.114.35", "12.207.114.41", fake.ipv4()])

    return {
        "alertName": "Internet Connection is Lost",
        "entityType": "service",
        "entityName": f"Adaptive Security Appliance 'outside' interface on {fw_host}",
        "alertSeverityString": "Emergency",
        "alertSeverity": 1,
        "triggerDescription": (
            f"The internet connection for this default gateway ({gateway_ip}) has been lost\r\n\r\n"
            "VPN/Direct connect to device in question\r\n"
            "Validate interface status\r\n"
            "Contact carrier and verify circuit status, if no LOA/support information, contact POC\r\n"
            "If status is good, Initiate Crisis management, warm handoff to IS - Networking"
        ),
        "clearDescription": f"The internet connection for this default gateway ({gateway_ip}) has been restored",
    }

def build_network_element_offline(customer):
    if customer["site"] == "dpOBT":
        entity_name = random.choice([
            "dpOBTlabsw01.dyopath.com",
            "dpOBTlabsw01.dyopath.com Member 2",
            "dyoOBTdepotsw03.dyopathcorp.com",
            "dyoOBTdepotsw02.dyopathcorp.com",
        ])
    elif customer["site"] == "slhvSC":
        entity_name = random.choice(["slhvSCsw02", "slhvSCsw01", "slhvSCdist01"])
    else:
        entity_name = fake.hostname()

    model = random.choice([
        "Cisco WS-C2960XR-48FPD-I",
        "Cisco WS-C3560X-24P-L",
        "Cisco WS-C3560X-24T-S",
        "Meraki MS225-48FP",
    ])

    return {
        "alertName": "Network Element Offline",
        "entityType": "device",
        "entityName": entity_name,
        "alertSeverityString": "Emergency",
        "alertSeverity": 1,
        "triggerDescription": (
            f"This network element has gone offline   {model}\r\n\r\n"
            "SD Tasks\r\n"
            "VPN into client/Log into Meraki portal depending on device\r\n"
            "Connect to nearest connected device (IE, if it's a server out, connect to connected switch to validate interface status)\r\n"
            "If it's up: problem with Auvik: Open support ticket with Auvik via Support option in lower right\r\n"
            "If down, verify ISP is showing site as up.\r\n"
            "If ISP shows down, begin crisis management, Validate power on device with client\r\n"
            "If power is good, warm handoff to IS - Networking team\r\n"
            "If you cannot contact the client, hold off on warm handoff until Power can be verified."
        ),
        "clearDescription": "Device Online Status is now equal to Online",
    }

builders = {
    "ap_offline": build_ap_offline,
    "interface_mismatch": build_interface_mismatch,
    "vpn_gateway_lost": build_vpn_gateway_lost,
    "internet_lost": build_internet_lost,
    "network_element_offline": build_network_element_offline,
}

alerts = []
active_alerts = []

current_time = datetime(2025, 11, 25, 14, 44, 0, tzinfo=timezone.utc)

for _ in range(500):
    current_time += timedelta(seconds=random.randint(5, 45), milliseconds=random.randint(0, 999))

    should_clear = active_alerts and random.random() < 0.28

    if should_clear:
        active = active_alerts.pop(random.randrange(len(active_alerts)))
        alert_id = encode_auvik_id(active["companyId"], make_object_id())

        alerts.append({
            "entityId": active["entityId"],
            "subject": "You have a new alert!",
            "alertStatusString": "Cleared",
            "alertId": alert_id,
            "alertName": active["alertName"],
            "entityName": active["entityName"],
            "companyName": active["companyName"],
            "entityType": active["entityType"],
            "date": format_auvik_ts(current_time),
            "link": f"https://{active['slug']}.us2.my.auvik.com/alert/{alert_id.split(',')[-1] if ',' in alert_id else make_object_id()}/summary",
            "alertStatus": 1,
            "correlationId": active["correlationId"],
            "alertDescription": active["clearDescription"],
            "alertSeverityString": active["alertSeverityString"],
            "alertSeverity": active["alertSeverity"],
            "companyId": active["companyId"],
        })
    else:
        customer = random.choice(customers)
        template = random.choice(customer["templates"])
        payload = builders[template](customer)

        entity_numeric_id = get_entity_numeric_id(customer["companyId"], payload["entityName"])
        entity_id = encode_auvik_id(customer["companyId"], entity_numeric_id)
        alert_object_id = make_object_id()
        alert_id = encode_auvik_id(customer["companyId"], alert_object_id)

        event = {
            "entityId": entity_id,
            "subject": "You have a new alert!",
            "alertStatusString": "Triggered",
            "alertId": alert_id,
            "alertName": payload["alertName"],
            "entityName": payload["entityName"],
            "companyName": customer["name"],
            "entityType": payload["entityType"],
            "date": format_auvik_ts(current_time),
            "link": f"https://{customer['slug']}.us2.my.auvik.com/alert/{alert_object_id}/summary",
            "alertStatus": 0,
            "correlationId": alert_id,
            "alertDescription": payload["triggerDescription"],
            "alertSeverityString": payload["alertSeverityString"],
            "alertSeverity": payload["alertSeverity"],
            "companyId": customer["companyId"],
        }

        alerts.append(event)
        active_alerts.append({
            "entityId": entity_id,
            "alertName": payload["alertName"],
            "entityName": payload["entityName"],
            "companyName": customer["name"],
            "entityType": payload["entityType"],
            "companyId": customer["companyId"],
            "slug": customer["slug"],
            "correlationId": alert_id,
            "alertSeverityString": payload["alertSeverityString"],
            "alertSeverity": payload["alertSeverity"],
            "clearDescription": payload["clearDescription"],
        })

with open("../../data/synthetic/auvik_synthetic.json", "w") as f:
    json.dump(alerts, f, indent=2)

### Meraki

In [9]:
def meraki_serial():
    chars = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
    part = lambda: "".join(random.choice(chars) for _ in range(4))
    return f"{part()}-{part()}-{part()}"

def meraki_token(n=8):
    chars = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    return "".join(random.choice(chars) for _ in range(n))

def utc_z(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

alert_catalog = [
    {"name": "VPN connectivity changed", "type_id": "vpn_connectivity_change", "level": "warning"},
    {"name": "Client IP conflict detected", "type_id": "ip_conflict", "level": "warning"},
    {"name": "Device offline", "type_id": "device_offline", "level": "critical"},
    {"name": "High latency", "type_id": "high_latency", "level": random.choice(["warning", "critical"])},
    {"name": "Port failure", "type_id": "port_failure", "level": random.choice(["warning", "critical"])},
    {"name": "High traffic", "type_id": "high_traffic", "level": random.choice(["warning", "critical"])},
]

org_profiles = []
for c in customers:
    org_id = str(fake.random_number(digits=18, fix_len=True))
    shard = random.randint(1, 200)
    org_token = meraki_token(8)
    app_key = fake.md5(raw_output=False)

    networks = []
    for _ in range(random.randint(1, 3)):
        n_token = meraki_token(8)
        net_slug = f"{c['site']}-{fake.word()}-{fake.word()}".replace(" ", "-")
        network_id = f"L_{fake.random_number(digits=18, fix_len=True)}"
        network_name = f" {c['site']} {fake.city()}"

        devices = []
        for _ in range(random.randint(1, 3)):
            device_name = f"{c['site']}-{random.choice(['FW', 'EDGE', 'MX'])}{random.randint(1, 99):02d}"
            device_model = random.choice(["MX68", "MX75", "MX85", "MX95", "MX100"])
            devices.append({
                "deviceSerial": meraki_serial(),
                "deviceMac": fake.mac_address(),
                "deviceName": device_name,
                "deviceModel": device_model,
            })

        networks.append({
            "networkId": network_id,
            "networkName": network_name,
            "networkUrl": f"https://n{shard}.dashboard.meraki.com/{net_slug}/n/{n_token}/manage/nodes/wired_status",
            "deviceUrlBase": f"https://n{shard}.dashboard.meraki.com/{net_slug}/n/{n_token}/manage/nodes",
            "alert_counter": random.randint(10_000_000, 99_999_999),
            "devices": devices,
        })

    org_profiles.append({
        "organizationId": org_id,
        "organizationName": c["name"],
        "organizationUrl": f"https://n{shard}.dashboard.meraki.com/o/{org_token}/manage/organization/overview",
        "app_key": app_key,
        "networks": networks,
    })

meraki_alerts = []
event_clock = datetime(2025, 11, 25, 17, 30, 0, tzinfo=timezone.utc)

while len(meraki_alerts) < 500:
    org = random.choice(org_profiles)
    net = random.choice(org["networks"])
    dev = random.choice(net["devices"])
    rule = random.choice(alert_catalog)

    occurred_at = event_clock + timedelta(seconds=random.randint(5, 60))
    copies = random.randint(1, 4)  # repeated notifications for same incident

    for _ in range(copies):
        if len(meraki_alerts) >= 500:
            break

        sent_at = occurred_at + timedelta(milliseconds=random.randint(200, 90000))
        net["alert_counter"] += 1
        alert_id = str(int(org["organizationId"]) + net["alert_counter"])

        level = rule["level"] if isinstance(rule["level"], str) else random.choice(["warning", "critical"])

        meraki_alerts.append({
            "app_key": org["app_key"],
            "status": level,
            "check": rule["name"],
            "version": "0.1",
            "sharedSecret": "",
            "sentAt": utc_z(sent_at),
            "organizationId": org["organizationId"],
            "organizationName": org["organizationName"],
            "organizationUrl": org["organizationUrl"],
            "networkId": net["networkId"],
            "networkName": net["networkName"],
            "networkUrl": net["networkUrl"],
            "deviceSerial": dev["deviceSerial"],
            "deviceMac": dev["deviceMac"],
            "deviceName": dev["deviceName"],
            "deviceUrl": f"{net['deviceUrlBase']}/new_wired_status",
            "deviceModel": dev["deviceModel"],
            "alertId": alert_id,
            "alertType": rule["name"],
            "alertTypeId": rule["type_id"],
            "alertLevel": level,
            "occurredAt": utc_z(occurred_at),
            "host": dev["deviceName"],
        })

    event_clock = occurred_at

with open("../../data/synthetic/meraki_synthetic.json", "w") as f:
    json.dump(meraki_alerts, f, indent=2)

### N-central

In [10]:
import xml.etree.ElementTree as ET

NCENTRAL_URI = "ncod524.n-able.com"

customer_profiles = [
    {
        "customerName": "Ocean Point Terminals (LTB)",
        "serviceOrg": "DYOPATH",
        "customerID": 164,
        "devices": [
            {"name": "LBTAMAG01", "ip": "10.108.6.90"},
            {"name": "LBTDC001", "ip": "10.108.1.1"},
            {"name": "HOVDC004", "ip": "10.108.1.58"},
            {"name": "HOVDC005", "ip": "10.108.6.37"},
        ],
    },
    {
        "customerName": "Wyffels",
        "serviceOrg": "DYOPATH",
        "customerID": 250,
        "devices": [
            {"name": "AmesUtility", "ip": "10.0.4.6"},
            {"name": "DC3", "ip": "10.0.1.24"},
        ],
    },
    {
        "customerName": "Wedgewood Pharmacy",
        "serviceOrg": "DYOPATH",
        "customerID": 351,
        "devices": [
            {"name": "NJ-EX01", "ip": "192.168.1.85"},
        ],
    },
    {
        "customerName": "Ambient Enterprises > Ambient Enterprises HCN (HAR)",
        "serviceOrg": "Ambient Enterprises",
        "customerID": 262,
        "devices": [
            {"name": "GlacierVmProd2", "ip": "10.10.6.6"},
        ],
    },
    {
        "customerName": "Signorelli Company",
        "serviceOrg": "DYOPATH",
        "customerID": 358,
        "devices": [
            {"name": "005-IGEL-UMS01", "ip": "10.5.6.6"},
        ],
    },
]

event_log_cases = [
    {
        "task": "AD (System Log)",
        "source": "NETLOGON",
        "category": "None",
        "event_id": "5807",
        "computer": "LBTDC001.LBTERMINALS.COM",
        "log_name": "System",
        "log_type": "warning",
        "description": (
            "During the past 4.10 hours there have been 3 connections to this Domain Controller "
            "from client machines whose IP addresses don't map to any of the existing sites in "
            "the enterprise. Those clients, therefore, have undefined sites and may connect to "
            "any Domain Controller including those that are in far distant locations from the "
            "clients."
        ),
    },
    {
        "task": "AD (System Log)",
        "source": "NETLOGON",
        "category": "None",
        "event_id": "5781",
        "computer": "HOVDC004.hovensa.com",
        "log_name": "System",
        "log_type": "warning",
        "description": (
            "Dynamic registration or deletion of one or more DNS records associated with DNS "
            "domain 'hovensa.com' failed."
        ),
    },
    {
        "task": "AD (System Log)",
        "source": "NETLOGON",
        "category": "None",
        "event_id": "5805",
        "computer": "LBTDC001.LBTERMINALS.COM",
        "log_name": "System",
        "log_type": "error",
        "description": (
            "The session setup from the computer LBCTXVDI5 failed to authenticate. "
            "The following error occurred: Access is denied."
        ),
    },
    {
        "task": "AD (Directory Service Log)",
        "source": "Microsoft-Windows-ActiveDirectory_DomainService",
        "category": "LDAP Interface",
        "event_id": "1216",
        "computer": "HOVDC005.hovensa.com",
        "log_name": "Directory Service",
        "log_type": "warning",
        "description": (
            "Internal event: An LDAP client connection was closed because of an error.\n\n"
            "Client IP:\n127.0.0.1:57310\n\n"
            "Additional Data\nError value:\n1236 The network connection was aborted by the local system."
        ),
    },
]

cpu_process_pool = [
    "MsMpEng",
    "SentinelAgent",
    "Rio",
    "TiWorker",
    "AutomationManager.AgentService",
    "minionhost",
    "QBDBMgrN",
    "QBW",
    "WmiPrvSE",
    "agent",
    "SentinelStaticEngineScanner",
]

memory_process_pool = [
    "tomcat10 (64 bit)",
    "elasticsearch-service-x64 (64 bit)",
    "SentinelAgent (64 bit)",
    "RMClient (64 bit)",
    "MsMpEng (64 bit)",
]

def rc_link(customer_id, device_id):
    return (
        f"https://{NCENTRAL_URI}:443/deepLinkAction.do"
        f"?method=deviceRC&customerID={customer_id}&deviceID={device_id}&language=en_US"
    )

def ts(dt):
    return dt.strftime("%Y-%m-%d %H:%M:%S")

def add_fields(notification, fields):
    for key, value in fields.items():
        ET.SubElement(notification, key).text = "" if value is None else str(value)

def build_cpu_block(usage):
    chosen = random.sample(cpu_process_pool, 5)
    lines = [f"CPU Usage: {usage:.2f} %"]
    for i, proc in enumerate(chosen, 1):
        lines.append(f"Top Process {i}: {proc}")
    for i in range(1, 6):
        lines.append(f"PID of Process {i}: {random.randint(1500, 32000)}")
    for i in range(1, 6):
        lines.append(
            f"User of Process {i}: {random.choice(['NT AUTHORITY\\SYSTEM', 'NT AUTHORITY\\LOCAL SERVICE', 'NT AUTHORITY\\NETWORK SERVICE'])}"
        )
    for i in range(1, 6):
        lines.append(f"CPU Usage for Process {i}: {random.randint(3, 40):.2f} %")
    return "\n".join(lines)

def build_memory_block():
    total_phys = round(random.uniform(15.5, 32.0), 2)
    used_phys = round(total_phys * random.uniform(0.88, 0.95), 2)
    free_phys = round(total_phys - used_phys, 2)
    total_virt = round(total_phys * random.uniform(1.2, 1.4), 2)
    used_virt = round(total_virt * random.uniform(0.75, 0.9), 2)
    free_virt = round(total_virt - used_virt, 2)
    lines = [
        f"Total Physical Memory: {total_phys:.2f} GB",
        f"Used Physical Memory: {used_phys:.2f} GB",
        f"Free Physical Memory: {free_phys:.2f} GB",
        f"Physical Memory Usage: {used_phys / total_phys * 100:.2f} %",
        f"Total Virtual Memory: {total_virt:.2f} GB",
        f"Used Virtual Memory: {used_virt:.2f} GB",
        f"Free Virtual Memory: {free_virt:.2f} GB",
        f"Virtual Memory Usage: {used_virt / total_virt * 100:.2f} %",
    ]
    chosen = random.sample(memory_process_pool, 5)
    for i, proc in enumerate(chosen, 1):
        lines.append(f"Top Process {i}: {proc}")
    for i in range(1, 6):
        lines.append(f"PID of Process {i}: {random.randint(1500, 32000)}")
    for i in range(1, 6):
        lines.append(
            f"User of Process {i}: {random.choice(['NT AUTHORITY\\SYSTEM', 'NT AUTHORITY\\LOCAL SERVICE', 'NT AUTHORITY\\NETWORK SERVICE'])}"
        )
    for i in range(1, 6):
        lines.append(f"Physical Memory for Process {i}: {random.uniform(250, 5200):.2f} MB")
    for i in range(1, 6):
        lines.append(f"Virtual Memory for Process {i}: {random.uniform(300, 6200):.2f} MB")
    return "\n".join(lines)

def build_disk_block():
    total = round(random.uniform(120, 500), 2)
    used_pct = random.randint(91, 97)
    used = round(total * used_pct / 100, 2)
    free = round(total - used, 2)
    return "\n".join([
        f"Total disk size: {total:.2f} GB",
        f"Disk space used: {used:.2f} GB",
        f"Disk free space: {free:.2f} GB",
        f"Disk Usage: {used_pct:.2f} %",
    ])

def build_event_log_block(case, event_dt):
    return "\n".join([
        "Event Log Module Status: 0",
        "The Last Record Number of the eventlog type that current event entry belongs to: 0",
        "# of duplicate events: 1.00 Events",
        f"Source: {case['source']}",
        f"Category: {case['category']}",
        f"Event ID: {case['event_id']}",
        "User (If Applicable): N/A",
        f"Computer: {case['computer']}",
        f"Event Description: {case['description']}",
        f"Event Log Name: {case['log_name']}",
        f"Event Log Type: {case['log_type']}",
        f"Event Log Date Time: {ts(event_dt)}",
    ])

def build_failed_notification(change_dt):
    template = random.choices(
        ["disk", "cpu", "eventlog", "exchange"],
        weights=[20, 25, 40, 15],
        k=1,
    )[0]

    if template == "disk":
        customer = customer_profiles[0]
        device = customer["devices"][0]
        return {
            "ActiveNotificationTriggerID": random.randint(600000000, 1400000000),
            "CustomerName": customer["customerName"],
            "DeviceURI": device["ip"],
            "DeviceName": device["name"],
            "ExternalCustomerID": "",
            "AffectedService": "Disk",
            "TaskIdent": "C:",
            "NcentralURI": NCENTRAL_URI,
            "QualitativeOldState": random.choice(["Warning", "Normal"]),
            "QualitativeNewState": "Failed",
            "TimeOfStateChange": ts(change_dt),
            "ProbeURI": device["ip"],
            "QuantitativeNewState": build_disk_block(),
            "ServiceOrganizationName": customer["serviceOrg"],
            "RemoteControlLink": rc_link(customer["customerID"], random.randint(100000000, 1999999999)),
        }

    if template == "cpu":
        customer = random.choice([customer_profiles[1], customer_profiles[3]])
        device = customer["devices"][0]
        return {
            "ActiveNotificationTriggerID": random.randint(600000000, 1400000000),
            "CustomerName": customer["customerName"],
            "DeviceURI": device["ip"],
            "DeviceName": device["name"],
            "ExternalCustomerID": "",
            "AffectedService": "CPU",
            "TaskIdent": "",
            "NcentralURI": NCENTRAL_URI,
            "QualitativeOldState": random.choice(["Warning", "Normal"]),
            "QualitativeNewState": "Failed",
            "TimeOfStateChange": ts(change_dt),
            "ProbeURI": device["ip"],
            "QuantitativeNewState": build_cpu_block(random.randint(94, 100)),
            "ServiceOrganizationName": customer["serviceOrg"],
            "RemoteControlLink": rc_link(customer["customerID"], random.randint(100000000, 1999999999)),
        }

    if template == "exchange":
        customer = customer_profiles[2]
        device = customer["devices"][0]
        return {
            "ActiveNotificationTriggerID": random.randint(600000000, 1400000000),
            "CustomerName": customer["customerName"],
            "DeviceURI": device["ip"],
            "DeviceName": device["name"],
            "ExternalCustomerID": "",
            "AffectedService": "Exchange Database 2016",
            "TaskIdent": "edgetransport",
            "NcentralURI": NCENTRAL_URI,
            "QualitativeOldState": "Normal",
            "QualitativeNewState": "Failed",
            "TimeOfStateChange": ts(change_dt),
            "ProbeURI": device["ip"],
            "QuantitativeNewState": "\n".join([
                "Database Page Fault Stalls: 0.00 Transactions/Second",
                "Database cache Percent Hit: 0.00 %",
                "Log Record Stalls : 0.00 Transactions/Second",
                "Log Threads Waiting: 0",
            ]),
            "ServiceOrganizationName": customer["serviceOrg"],
            "RemoteControlLink": rc_link(customer["customerID"], random.randint(100000000, 1999999999)),
        }

    customer = customer_profiles[0]
    device = random.choice(customer["devices"][1:])
    case = random.choice(event_log_cases)
    event_dt = change_dt + timedelta(minutes=random.randint(60, 130))
    return {
        "ActiveNotificationTriggerID": random.randint(600000000, 1400000000),
        "CustomerName": customer["customerName"],
        "DeviceURI": device["ip"],
        "DeviceName": device["name"],
        "ExternalCustomerID": "",
        "AffectedService": "Windows Event Log",
        "TaskIdent": case["task"],
        "NcentralURI": NCENTRAL_URI,
        "QualitativeOldState": "Normal",
        "QualitativeNewState": "Failed",
        "TimeOfStateChange": ts(change_dt),
        "ProbeURI": device["ip"],
        "QuantitativeNewState": build_event_log_block(case, event_dt),
        "ServiceOrganizationName": customer["serviceOrg"],
        "RemoteControlLink": rc_link(customer["customerID"], random.randint(100000000, 1999999999)),
    }

def build_clear_notification(change_dt):
    template = random.choices(
        ["agent", "connectivity", "cpu", "memory"],
        weights=[20, 20, 35, 25],
        k=1,
    )[0]

    if template == "agent":
        customer = customer_profiles[1]
        device = customer["devices"][0]
        return {
            "CustomerName": customer["customerName"],
            "DeviceURI": device["ip"],
            "DeviceName": device["name"],
            "AffectedService": "Agent Status",
            "TaskIdent": "",
            "QualitativeOldState": "Failed",
            "QualitativeNewState": "Normal",
            "TimeOfStateChange": ts(change_dt),
            "ProbeURI": NCENTRAL_URI,
            "QuantitativeNewState": "Agent check-in interval: 0.00 sec",
            "RemoteControlLink": rc_link(customer["customerID"], random.randint(100000000, 1999999999)),
            "AcknowledgementTime": "[no data available in DB]",
            "AcknowledgementUser": "[no data available in DB]",
            "ActiveProfile": "BASELINE - Server Agent - Notification",
        }

    if template == "connectivity":
        customer = customer_profiles[1]
        device = customer["devices"][0]
        return {
            "CustomerName": customer["customerName"],
            "DeviceURI": device["ip"],
            "DeviceName": device["name"],
            "AffectedService": "Connectivity",
            "TaskIdent": "",
            "QualitativeOldState": "Failed",
            "QualitativeNewState": "Normal",
            "TimeOfStateChange": ts(change_dt),
            "ProbeURI": "10.0.0.26",
            "QuantitativeNewState": "\n".join([
                "Packet Loss: 0.00 %",
                "Time To Live: 124.00 Hops",
                "Average Round Trip Time: 17.00 msec",
                "DNS Resolution: True",
            ]),
            "RemoteControlLink": rc_link(customer["customerID"], random.randint(100000000, 1999999999)),
            "AcknowledgementTime": "[no data available in DB]",
            "AcknowledgementUser": "[no data available in DB]",
            "ActiveProfile": "BASELINE - Server Connectivity - Notification",
        }

    if template == "memory":
        customer = customer_profiles[4]
        device = customer["devices"][0]
        return {
            "CustomerName": customer["customerName"],
            "DeviceURI": device["ip"],
            "DeviceName": device["name"],
            "AffectedService": "Memory",
            "TaskIdent": "",
            "QualitativeOldState": "Failed",
            "QualitativeNewState": "Normal",
            "TimeOfStateChange": ts(change_dt),
            "ProbeURI": device["ip"],
            "QuantitativeNewState": build_memory_block(),
            "RemoteControlLink": rc_link(customer["customerID"], random.randint(100000000, 1999999999)),
            "AcknowledgementTime": "[no data available in DB]",
            "AcknowledgementUser": "[no data available in DB]",
            "ActiveProfile": "BASELINE - Server Memory - Notification",
        }

    customer = random.choice([customer_profiles[1], customer_profiles[3]])
    device = customer["devices"][0]
    return {
        "CustomerName": customer["customerName"],
        "DeviceURI": device["ip"],
        "DeviceName": device["name"],
        "AffectedService": "CPU",
        "TaskIdent": "",
        "QualitativeOldState": "Failed",
        "QualitativeNewState": "Normal",
        "TimeOfStateChange": ts(change_dt),
        "ProbeURI": device["ip"],
        "QuantitativeNewState": build_cpu_block(random.randint(80, 88)),
        "RemoteControlLink": rc_link(customer["customerID"], random.randint(100000000, 1999999999)),
        "AcknowledgementTime": "[no data available in DB]",
        "AcknowledgementUser": "[no data available in DB]",
        "ActiveProfile": "BASELINE - Server CPU - Notification",
    }

docs = []
ncentral_time = datetime(2025, 11, 25, 8, 15, 0)

for _ in range(300):
    ncentral_time += timedelta(seconds=random.randint(30, 240))
    notification = ET.Element("notification")

    payload = (
        build_failed_notification(ncentral_time)
        if random.random() < 0.62
        else build_clear_notification(ncentral_time)
    )

    add_fields(notification, payload)

    try:
        ET.indent(notification, space="  ")
    except Exception:
        pass

    docs.append(ET.tostring(notification, encoding="unicode"))

with open("../../data/synthetic/ncentral_synthetic.xml", "w", encoding="utf-8") as f:
    for doc in docs:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write(doc)
        f.write("\n\n")

In [8]:
import json
import random
from datetime import datetime, timedelta

INPUT_FILE = "../../data/raw/Meraki Payload.json"
OUTPUT_FILE = "../../data/raw/meraki_augmented.json"

MIN_RESOLUTION = 5
MAX_RESOLUTION = 60
RESOLUTION_RATE = 0.7


def generate_resolution(alert):

    start = datetime.fromisoformat(alert["occurredAt"].replace("Z",""))
    end = start + timedelta(minutes=random.randint(MIN_RESOLUTION, MAX_RESOLUTION))

    resolved = alert.copy()

    resolved["alertId"] = alert["alertId"] + "_resolved"
    resolved["occurredAt"] = end.isoformat() + "Z"
    resolved["sentAt"] = end.isoformat() + "Z"

    resolved["status"] = "ok"
    resolved["alertLevel"] = "info"

    resolved["eventState"] = "resolved"
    resolved["synthetic"] = True
    resolved["correlationId"] = alert["alertId"]

    return resolved


def main():

    with open(INPUT_FILE) as f:
        alerts = json.load(f)

    output = []

    for alert in alerts:

        alert["eventState"] = "triggered"
        alert["synthetic"] = False
        alert["correlationId"] = alert["alertId"]

        output.append(alert)

        if random.random() < RESOLUTION_RATE:
            output.append(generate_resolution(alert))

    output.sort(key=lambda x: x["occurredAt"])

    with open(OUTPUT_FILE,"w") as f:
        json.dump(output,f,indent=2)

    print("Generated alerts:",len(output))


In [14]:
main()

Generated alerts: 13


In [17]:
import psycopg2
import random
from faker import Faker
from datetime import datetime, timedelta
import uuid

fake = Faker()

TOTAL_ALERTS = 5000

conn = psycopg2.connect(
    host="localhost",
    database="kenexaihackathon",
    user="postgres",
    password="09092002"
)

cursor = conn.cursor()


def random_time():
    start = datetime(2025,1,1)
    return start + timedelta(minutes=random.randint(0,500000))


def generate_meraki():

    device = fake.hostname()
    alert_id = str(uuid.uuid4())
    t = random_time()

    cursor.execute("""
    INSERT INTO bronze.meraki_alerts
    (
        app_key,status,check_name,version,
        sent_at,occurred_at,
        organization_id,organization_name,
        network_id,network_name,
        device_serial,device_mac,device_name,
        device_model,host,
        alert_id,alert_type,alert_type_id,
        alert_level,event_state,synthetic,
        correlation_id
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """,
    (
        fake.uuid4(),
        "warning",
        "VPN connectivity changed",
        "0.1",
        t,
        t,
        fake.uuid4(),
        fake.company(),
        fake.uuid4(),
        fake.city(),
        fake.uuid4(),
        fake.mac_address(),
        device,
        "MX68",
        device,
        alert_id,
        "VPN connectivity changed",
        "vpn_connectivity_change",
        "warning",
        "triggered",
        False,
        alert_id
    ))

    if random.random() < 0.7:

        resolved_time = t + timedelta(minutes=random.randint(5,60))

        cursor.execute("""
        INSERT INTO bronze.meraki_alerts
        (
            app_key,status,check_name,version,
            sent_at,occurred_at,
            organization_id,organization_name,
            network_id,network_name,
            device_serial,device_mac,device_name,
            device_model,host,
            alert_id,alert_type,alert_type_id,
            alert_level,event_state,synthetic,
            correlation_id
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """,
        (
            fake.uuid4(),
            "ok",
            "VPN connectivity changed",
            "0.1",
            resolved_time,
            resolved_time,
            fake.uuid4(),
            fake.company(),
            fake.uuid4(),
            fake.city(),
            fake.uuid4(),
            fake.mac_address(),
            device,
            "MX68",
            device,
            alert_id+"_resolved",
            "VPN connectivity restored",
            "vpn_connectivity_change",
            "info",
            "resolved",
            True,
            alert_id
        ))


def generate_auvik():

    device = fake.hostname()
    alert_id = str(uuid.uuid4())
    t = random_time()

    cursor.execute("""
    INSERT INTO bronze.auvik_alerts
    (
        entity_id,subject,alert_status_string,
        alert_id,alert_name,
        entity_name,company_name,
        entity_type,alert_description,
        alert_severity_string,
        alert_severity,
        correlation_id,
        alert_status,
        event_time,
        event_state,
        synthetic
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """,
    (
        fake.uuid4(),
        "You have a new alert",
        "Triggered",
        alert_id,
        "Device Offline",
        device,
        fake.company(),
        "device",
        "Device unreachable",
        "Critical",
        2,
        alert_id,
        0,
        t,
        "triggered",
        False
    ))

    if random.random() < 0.7:

        resolved_time = t + timedelta(minutes=random.randint(5,60))

        cursor.execute("""
        INSERT INTO bronze.auvik_alerts
        (
            entity_id,subject,alert_status_string,
            alert_id,alert_name,
            entity_name,company_name,
            entity_type,alert_description,
            alert_severity_string,
            alert_severity,
            correlation_id,
            alert_status,
            event_time,
            event_state,
            synthetic
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """,
        (
            fake.uuid4(),
            "Alert cleared",
            "Cleared",
            alert_id+"_resolved",
            "Device Offline",
            device,
            fake.company(),
            "device",
            "Device reachable again",
            "Info",
            5,
            alert_id,
            1,
            resolved_time,
            "resolved",
            True
        ))


def generate_ncentral():

    device = fake.hostname()
    alert_id = str(uuid.uuid4())
    t = random_time()

    cursor.execute("""
    INSERT INTO bronze.ncentral_alerts
    (
        active_notification_trigger_id,
        customer_name,
        device_uri,
        device_name,
        affected_service,
        qualitative_old_state,
        qualitative_new_state,
        time_of_state_change,
        event_state,
        synthetic,
        correlation_id
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """,
    (
        alert_id,
        fake.company(),
        fake.ipv4(),
        device,
        "CPU",
        "Normal",
        "Failed",
        t,
        "triggered",
        False,
        alert_id
    ))

    if random.random() < 0.7:

        resolved_time = t + timedelta(minutes=random.randint(5,60))

        cursor.execute("""
        INSERT INTO bronze.ncentral_alerts
        (
            active_notification_trigger_id,
            customer_name,
            device_uri,
            device_name,
            affected_service,
            qualitative_old_state,
            qualitative_new_state,
            time_of_state_change,
            event_state,
            synthetic,
            correlation_id
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """,
        (
            alert_id+"_resolved",
            fake.company(),
            fake.ipv4(),
            device,
            "CPU",
            "Failed",
            "Normal",
            resolved_time,
            "resolved",
            True,
            alert_id
        ))


for i in range(TOTAL_ALERTS):

    generate_meraki()
    generate_auvik()
    generate_ncentral()

    if i % 1000 == 0:
        print("Generated", i)

conn.commit()

cursor.close()
conn.close()

print("Synthetic dataset generation complete.")


Generated 0
Generated 1000
Generated 2000
Generated 3000
Generated 4000
Synthetic dataset generation complete.
